In [59]:
 %pip -q install pandas datasets scikit-learn

In [60]:
from datasets import load_dataset

dataset = load_dataset("albaz2000/arabic-itsm-dataset")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['ticket_id', 'created_at', 'updated_at', 'channel', 'model', 'dialect', 'title_ar', 'description_ar', 'category_level_1', 'category_level_2', 'category_level_3', 'category_path', 'tags', 'labels_json', 'impact', 'urgency', 'priority', 'sentiment'],
        num_rows: 10000
    })
})


In [61]:
print("أقسام البيانات:")
print(dataset.keys())

split_name = list(dataset.keys())[0]
data = dataset[split_name]

print("\nعدد السجلات:", len(data))
print("أسماء الأعمدة:", data.column_names)

print("\nأول سجل:")
print(data[0])

أقسام البيانات:
dict_keys(['train'])

عدد السجلات: 10000
أسماء الأعمدة: ['ticket_id', 'created_at', 'updated_at', 'channel', 'model', 'dialect', 'title_ar', 'description_ar', 'category_level_1', 'category_level_2', 'category_level_3', 'category_path', 'tags', 'labels_json', 'impact', 'urgency', 'priority', 'sentiment']

أول سجل:
{'ticket_id': 'TCKT-001-001', 'created_at': '2026-02-04T22:41:00+02:00', 'updated_at': '2026-02-05T02:41:00+02:00', 'channel': 'email', 'model': 'gemini-3-flash', 'dialect': 'Egyptian', 'title_ar': 'السيستم واقع', 'description_ar': 'يا جماعة السيستم كله مش شغال عند كل الناس في القسم، في عطل عام؟. بقالها 21 ساعة تقريبا. لو محتاجين لوجز أو سكرينشوت أنا جاهز أبعتها.', 'category_level_1': 'Service', 'category_level_2': 'Incident', 'category_level_3': 'Outage', 'category_path': 'Service > Incident > Outage', 'tags': ['outage', 'system'], 'labels_json': {'l1': 'Service', 'l2': 'Incident', 'l3': 'Outage', 'tags': ['outage', 'down', 'system']}, 'impact': 3, 'urgency': 

In [62]:
import pandas as pd

# تحويل البيانات إلى جدول يسهل تحليله
df = dataset["train"].to_pandas()

# الأعمدة الأساسية التي سنستخدمها
text_columns = ["title_ar", "description_ar"]
target_columns = [
    "category_level_1",
    "category_level_2",
    "category_level_3",
    "priority",
    "sentiment"
]

print("1) حجم البيانات:", df.shape)

# فحص القيم المفقودة
print("\n2) القيم المفقودة في الأعمدة المهمة:")
print(df[text_columns + target_columns].isna().sum())

# فحص التصنيفات
for col in target_columns:
    print(f"\n3) توزيع {col}:")
    print(df[col].value_counts(dropna=False).head(20))

# فحص اللهجات
print("\n4) توزيع اللهجات:")
print(df["dialect"].value_counts(dropna=False))

# فحص التذاكر المتكررة
print("\n5) عدد العناوين المكررة:")
print(df["title_ar"].duplicated().sum())

print("\nعدد أوصاف التذاكر المكررة:")
print(df["description_ar"].duplicated().sum())

# عرض أمثلة متنوعة
print("\n6) خمس تذاكر مع تصنيفاتها:")
display(
    df[
        ["title_ar", "description_ar",
         "category_level_1", "category_level_2",
         "category_level_3", "priority"]
    ].sample(5, random_state=42)
)

1) حجم البيانات: (10000, 18)

2) القيم المفقودة في الأعمدة المهمة:
title_ar            0
description_ar      0
category_level_1    0
category_level_2    0
category_level_3    0
priority            0
sentiment           0
dtype: int64

3) توزيع category_level_1:
category_level_1
Access      1987
Hardware    1869
Software    1846
Network     1820
Service     1275
Security    1203
Name: count, dtype: int64

3) توزيع category_level_2:
category_level_2
Account              763
Request              675
Laptop/Desktop       657
Permissions          643
VPN                  642
Printer/Scanner      628
Office Apps          620
Malware/Phishing     619
Email/Calendar       618
WiFi                 613
Business App         608
Incident             600
Policy/Compliance    584
Peripherals          584
MFA/SSO              581
Internet/LAN         565
Name: count, dtype: int64

3) توزيع category_level_3:
category_level_3
Password Reset        275
Account Locked        269
New Device            254

,title_ar,description_ar,category_level_1,category_level_2,category_level_3,priority
6252,كيبورد بايظة - 253,في زراير في الكيبورد مش شغالة خالص. بقالها 7 س...,Access,Account,Password Reset,4
4684,مشكلة في Keyboard/Mouse,يا جماعة الـ Keyboard/Mouse مش شغال خالص وبقال...,Hardware,Peripherals,Keyboard/Mouse,2
1731,مفيش إذن دخول,كل ما أفتح الملف بيقولي Access Denied، مع إني ...,Access,Permissions,Permission Denied,4
4742,طلب بخصوص Device Encryption,لو سمحتوا محتاج مساعدة في الـ Device Encryptio...,Security,Policy/Compliance,Device Encryption,4
4521,بطل يشتغل Authenticator Issue,الـ Authenticator Issue فجأة كده مابقاش يفتح، ...,Access,MFA/SSO,Authenticator Issue,2


In [63]:
# 1. هل توجد تذاكر متطابقة تمامًا؟
text_cols = ["title_ar", "description_ar"]
label_col = "category_level_1"

print("عدد التذاكر المتطابقة في النصين:")
print(df.duplicated(subset=text_cols).sum())

# 2. هل النص نفسه مرتبط بأكثر من تصنيف؟
text_label_counts = (
    df.groupby(text_cols, dropna=False)[label_col]
      .nunique()
)

print("\nعدد مجموعات النصوص المرتبطة بأكثر من تصنيف:")
print((text_label_counts > 1).sum())

# 3. ما أكثر العناوين تكرارًا، وما تصنيفاتها؟
title_counts = df["title_ar"].value_counts()

print("\nأكثر 10 عناوين تكرارًا:")
display(title_counts.head(10).to_frame("count"))

# 4. فحص التصنيفات المرتبطة بالعناوين المتكررة
frequent_titles = title_counts[title_counts >= 5].index
repeated = df[df["title_ar"].isin(frequent_titles)]

title_label_summary = (
    repeated.groupby("title_ar")
    .agg(
        count=("ticket_id", "size"),
        categories=("category_level_1",
                    lambda x: x.nunique()),
        examples=("category_level_1",
                  lambda x: ", ".join(x.unique()))
    )
    .sort_values("count", ascending=False)
)

print("\nعناوين متكررة مرتبطة بتصنيفات مختلفة:")
display(
    title_label_summary[
        title_label_summary["categories"] > 1
    ].head(15)
)

# 5. مراجعة عينات من التذاكر لتقييم جودة التصنيف
display(
    df.sample(15, random_state=7)[
        ["title_ar", "description_ar",
         "category_level_1", "category_level_2",
         "category_level_3"]
    ]
)

عدد التذاكر المتطابقة في النصين:
451

عدد مجموعات النصوص المرتبطة بأكثر من تصنيف:
43

أكثر 10 عناوين تكرارًا:


,count
title_ar,
مشكلة فنية,127
طلب دعم,121
عطل في النظام,112
طلب لابتوب جديد,85
مشكلة في الدخول,82
طلب صلاحيات,78
الـ VPN مش شغال,69
تغيير كلمة السر,65
نسيت الباسورد,64



عناوين متكررة مرتبطة بتصنيفات مختلفة:


,count,categories,examples
title_ar,,,
مشكلة فنية,127,6,"Network, Software, Service, Hardware, Access, ..."
طلب دعم,121,6,"Access, Software, Security, Network, Hardware,..."
عطل في النظام,112,6,"Access, Service, Software, Hardware, Network, ..."
طلب لابتوب جديد,85,6,"Service, Security, Hardware, Access, Network, ..."
مشكلة في الدخول,82,6,"Access, Hardware, Software, Service, Security,..."
طلب صلاحيات,78,6,"Access, Service, Security, Hardware, Network, ..."
الـ VPN مش شغال,69,6,"Network, Software, Access, Service, Hardware, ..."
نسيت الباسورد,64,5,"Access, Service, Network, Software, Hardware"
النت بطيء,48,6,"Network, Hardware, Security, Service, Software..."


,title_ar,description_ar,category_level_1,category_level_2,category_level_3
1977,محاولة احتيال,في إيميل مبعوت من عنوان غريب بيقولي كسبت جايزة...,Security,Malware/Phishing,Phishing Email
3880,صوت المروحة عالي,الويندوز طالب ابديت بس انا خايف اعمله الجهاز ي...,Security,Malware/Phishing,Phishing Email
52,مشكلة في السرعة,البي سي بتاعي بطيء جداً ومحتاج يتعمله فورمات أ...,Hardware,Laptop/Desktop,Performance
2551,مشكلة في الربط,اللابتوب مش راضي يلقط واي فاي الشركة خالص. على...,Network,WiFi,Authentication
2246,سرعة النت وحشة,السرعة النهاردة مش معقولة، الواي فاي تقيل جداً...,Network,WiFi,Slow Speed
270,مشكلة في السينك,الإيميلات مش بتوصلني على الأوتلوك مع إنها موجو...,Software,Email/Calendar,Outlook Issue
601,مشكلة في Role Request,في عطل مفاجئ حصل ومحتاج حد يكلمني ضروري. بخصوص...,Access,Permissions,Role Request
2441,طلب وصول لبيانات حساسة,محتاج أكسس على داتا قسم الموارد البشرية. بقاله...,Security,Policy/Compliance,Data Access
3286,خطأ في النطاق,يا ريت الرد بسرعة عشان فيه تسليمات مهمة النهار...,Network,Internet/LAN,DNS
2967,مشكلة في البطارية,البطارية مش بتشحن خالص، لازم يفضل متوصل بالشاح...,Hardware,Laptop/Desktop,Performance


In [64]:
# تجميع النصوص المتطابقة المرتبطة بأكثر من تصنيف رئيسي
text_cols = ["title_ar", "description_ar"]

conflict_keys = (
    df.groupby(text_cols, dropna=False)["category_level_1"]
      .nunique()
)

conflict_keys = conflict_keys[conflict_keys > 1].index

conflicts = (
    df.set_index(text_cols)
      .loc[conflict_keys]
      .reset_index()
)

print("عدد السجلات ضمن مجموعات النصوص المتعارضة:",
      len(conflicts))

display(
    conflicts[
        ["title_ar", "description_ar", "category_level_1",
         "category_level_2", "category_level_3"]
    ].head(20)
)

عدد السجلات ضمن مجموعات النصوص المتعارضة: 137


,title_ar,description_ar,category_level_1,category_level_2,category_level_3
0,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Software,Business App,Integration
1,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Hardware,Laptop/Desktop,Battery
2,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Hardware,Peripherals,Docking Station
3,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Network,Internet/LAN,Latency
4,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Security,Policy/Compliance,Blocked Site
5,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Access,Permissions,Admin Access
6,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Service,Incident,Intermittent
7,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Access,MFA/SSO,SSO Login Issue
8,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Software,Email/Calendar,Mailbox Access
9,طلب دعم,التطبيق مش شغال زي العادة وفيه أخطاء بتظهر. بق...,Access,MFA/SSO,Authenticator Issue


In [65]:
text_cols = ["title_ar", "description_ar"]

# عدد مجموعات النصوص المتعارضة
conflict_groups = (
    df.groupby(text_cols, dropna=False)["category_level_1"]
      .nunique()
)

conflict_groups = conflict_groups[conflict_groups > 1]

print("عدد مجموعات النصوص المتعارضة:", len(conflict_groups))
print("عدد السجلات ضمن هذه المجموعات:", len(conflicts))

# توزيع التصنيفات داخل السجلات المتعارضة
print("\nتكرار التصنيفات المتعارضة:")
print(conflicts["category_level_1"].value_counts())


عدد مجموعات النصوص المتعارضة: 43
عدد السجلات ضمن هذه المجموعات: 137

تكرار التصنيفات المتعارضة:
category_level_1
Hardware    33
Security    28
Service     26
Access      22
Software    17
Network     11
Name: count, dtype: int64


In [66]:
# 1. تحديد النصوص التي ترتبط بأكثر من تصنيف رئيسي
text_cols = ["title_ar", "description_ar"]

label_counts = (
    df.groupby(text_cols, dropna=False)["category_level_1"]
      .nunique()
)

conflict_keys = label_counts[label_counts > 1].index

# 2. استبعاد كل السجلات التي تنتمي إلى مجموعات متعارضة
clean_df = df.set_index(text_cols)

clean_df = clean_df.loc[
    ~clean_df.index.isin(conflict_keys)
].reset_index()

# 3. إزالة التكرارات المطابقة في النص والتصنيف
clean_df = clean_df.drop_duplicates(
    subset=["title_ar", "description_ar", "category_level_1"]
).reset_index(drop=True)

print("عدد السجلات الأصلية:", len(df))
print("عدد السجلات بعد التنظيف:", len(clean_df))
print("عدد السجلات المستبعدة:", len(df) - len(clean_df))

print("\nتوزيع التصنيفات بعد التنظيف:")
print(clean_df["category_level_1"].value_counts())

عدد السجلات الأصلية: 10000
عدد السجلات بعد التنظيف: 9506
عدد السجلات المستبعدة: 494

توزيع التصنيفات بعد التنظيف:
category_level_1
Access      1897
Software    1767
Hardware    1756
Network     1752
Service     1202
Security    1132
Name: count, dtype: int64


In [67]:
text_cols = ["title_ar", "description_ar"]

remaining_conflicts = (
    clean_df.groupby(text_cols, dropna=False)["category_level_1"]
            .nunique()
)

print(
    "مجموعات النصوص المتعارضة المتبقية:",
    (remaining_conflicts > 1).sum()
)

print(
    "عدد النصوص المكررة المتبقية:",
    clean_df.duplicated(subset=text_cols).sum()
)

مجموعات النصوص المتعارضة المتبقية: 0
عدد النصوص المكررة المتبقية: 0


In [68]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# دمج عنوان التذكرة ووصفها
X = (
    clean_df["title_ar"].fillna("") + " " +
    clean_df["description_ar"].fillna("")
)

# التصنيف الذي نريد أن يتعلم النموذج التنبؤ به
y = clean_df["category_level_1"]

# تقسيم البيانات مع الحفاظ على نسب التصنيفات
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("عدد سجلات التدريب:", len(X_train))
print("عدد سجلات الاختبار:", len(X_test))

# إنشاء النموذج وتدريبه
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=30000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# تقييم النموذج على بيانات الاختبار
y_pred = model.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

عدد سجلات التدريب: 7604
عدد سجلات الاختبار: 1902

Accuracy: 0.8838

Classification Report:

              precision    recall  f1-score   support

      Access       0.85      0.88      0.86       380
    Hardware       0.87      0.87      0.87       351
     Network       0.99      0.92      0.95       351
    Security       0.82      0.87      0.84       226
     Service       0.77      0.88      0.82       240
    Software       1.00      0.88      0.93       354

    accuracy                           0.88      1902
   macro avg       0.88      0.88      0.88      1902
weighted avg       0.89      0.88      0.89      1902



In [69]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=[f"الحقيقي: {label}" for label in labels],
    columns=[f"توقع: {label}" for label in labels]
)

display(cm_df)

,توقع: Access,توقع: Hardware,توقع: Network,توقع: Security,توقع: Service,توقع: Software
الحقيقي: Access,336,11,1,11,21,0
الحقيقي: Hardware,11,305,2,17,16,0
الحقيقي: Network,9,7,322,2,10,1
الحقيقي: Security,11,7,0,196,12,0
الحقيقي: Service,14,7,1,7,211,0
الحقيقي: Software,16,15,0,7,5,311


In [70]:
# جمع نتائج الاختبار مع التوقعات
errors_df = pd.DataFrame({
    "النص": X_test.values,
    "التصنيف_الحقيقي": y_test.values,
    "توقع_النموذج": y_pred
})

# الاحتفاظ بالسجلات التي أخطأ النموذج فيها
errors_df = errors_df[
    errors_df["التصنيف_الحقيقي"] != errors_df["توقع_النموذج"]
]

# عرض عينة عشوائية قابلة لإعادة الإنتاج
display(
    errors_df.sample(
        n=min(15, len(errors_df)),
        random_state=42
    ).reset_index(drop=True)
)

,النص,التصنيف_الحقيقي,توقع_النموذج
0,تحديث الاوفيس الاوفيس محتاج تفعيل وكل شوية تطل...,Software,Security
1,السيستم واقع - 467 السيستم مش بيفتح خالص عند ك...,Hardware,Service
2,البطارية بتخلص النت في اوضة الاجتماعات ضعيف جد...,Network,Hardware
3,مشكلة فنية لو سمحتوا حد يشوفلي حل للمشكلة دي ع...,Security,Service
4,السيستم واقع - 410 السيستم مش بيفتح خالص عند ك...,Access,Service
5,تحديث الويندوز اللابتوب بيسخن جداً وبيهنج اول ...,Hardware,Security
6,طلب برنامج - 337 محتاج أنزل برنامج جديد للشغل،...,Access,Service
7,النت بطيء النت فاصل خالص في المكتب ومفيش اي اش...,Network,Service
8,البطارية بتخلص شاحن اللابتوب باظ ومحتاج واحد ج...,Service,Hardware
9,الايميل مش شغال الايميلات مش بتوصلني خالص من ا...,Access,Service


In [71]:
# الاحتفاظ بالنص الأصلي لكل سجل اختبار
error_details = clean_df.loc[
    X_test.index if hasattr(X_test, "index") else []
] if False else None

# إعادة بناء تفاصيل الاختبار من الفهارس الأصلية
X_all = (
    clean_df["title_ar"].fillna("") + " " +
    clean_df["description_ar"].fillna("")
)

# استخدام فهارس مجموعة الاختبار
test_indices = X_test.index

error_details = clean_df.loc[test_indices, [
    "title_ar",
    "description_ar",
    "category_level_1"
]].copy()

error_details["التوقع"] = y_pred
error_details = error_details[
    error_details["category_level_1"] != error_details["التوقع"]
]

display(error_details.head(10))

,title_ar,description_ar,category_level_1,التوقع
3505,طلب شاشة اضافية,عايز اطلب صلاحيات ادمن عشان اسطب درايفر للبرين...,Access,Hardware
6098,طلب برنامج - 397,محتاج أنزل برنامج جديد للشغل، ممكن تساعدوني؟. ...,Access,Service
6673,الموقع محجوب,الصوت في الاجتماع مش واضح خالص وفي وش كتير، جر...,Network,Security
3647,النت مقطوع,كود الـ MFA مش بيوصلي على الموبايل، انا واقف د...,Security,Access
3253,الورق محشور,أنا لسه مستلم الجهاز والظاهر إن فيه حاجة مش مت...,Hardware,Security
3443,الـ SSO فيه مشكلة,يا جماعة انا مش عارف ادخل على الاكونت بتاعي خا...,Service,Hardware
6047,السيستم واقع - 346,السيستم مش بيفتح خالص عند كل الناس في القسم. ب...,Hardware,Service
5022,عطل في مكنة التصوير,لو سمحتوا حد يشوفلي حل للمشكلة دي عشان متعطل. ...,Hardware,Service
2931,اقتراح تطوير,جربت أعمل ريستارت أكتر من مرة ومازال الوضع زي ...,Software,Hardware
3583,فيروس على الجهاز,شاحن اللابتوب باظ ومحتاج واحد جديد مكانه بسرعة...,Access,Security


In [72]:
knowledge_base = [
    {
        "problem": "المستخدم لا يستطيع تسجيل الدخول بسبب نسيان كلمة المرور",
        "solution": "استخدم خيار إعادة تعيين كلمة المرور من البوابة الرسمية. إذا استمرت المشكلة، تواصل مع فريق الدعم للتحقق من الحساب."
    },
    {
        "problem": "رمز المصادقة متعددة العوامل MFA لا يصل إلى الهاتف",
        "solution": "تحقق من اتصال الهاتف بالشبكة ومن إعدادات تطبيق المصادقة. لا تشارك رمز التحقق مع أي شخص. إذا استمرت المشكلة، تواصل مع الدعم الفني."
    },
    {
        "problem": "الاتصال بشبكة Wi-Fi الخاصة بالشركة لا يعمل",
        "solution": "تحقق من تفعيل Wi-Fi واختيار الشبكة الصحيحة. جرّب إعادة الاتصال، وإذا استمرت المشكلة زوّد الدعم باسم الشبكة ورسالة الخطأ."
    },
    {
        "problem": "الطابعة لا تطبع أو تظهر رسالة خطأ",
        "solution": "تحقق من تشغيل الطابعة واتصالها، ومن وجود الورق وعدم انحشاره. إذا استمرت المشكلة، أرسل للدعم رسالة الخطأ واسم الطابعة."
    },
    {
        "problem": "رسائل البريد الإلكتروني لا تصل إلى Outlook",
        "solution": "تحقق من اتصال الإنترنت وحالة صندوق البريد، ثم أعد تشغيل التطبيق. إذا استمرت المشكلة، تواصل مع الدعم مع ذكر وقت حدوثها."
    },
    {
        "problem": "الاشتباه برسالة بريد إلكتروني تصيدية",
        "solution": "لا تضغط على الروابط أو تفتح المرفقات، ولا تدخل بياناتك. أبلغ فريق الأمن عبر القناة الرسمية واتبع سياسة المؤسسة."
    }
]

print("عدد الحلول في قاعدة المعرفة:", len(knowledge_base))

عدد الحلول في قاعدة المعرفة: 6


In [73]:
!pip -q install sentence-transformers

In [74]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# تحميل نموذج التضمينات متعدد اللغات
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# النصوص التي سيبحث النموذج بينها
knowledge_texts = [
    item["problem"] for item in knowledge_base
]

# تحويل المشكلات إلى متجهات رقمية
knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    normalize_embeddings=True
)

print("عدد المشكلات:", len(knowledge_texts))
print("شكل المتجهات:", knowledge_embeddings.shape)
print("تم إنشاء Embeddings بنجاح")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

عدد المشكلات: 6
شكل المتجهات: (6, 384)
تم إنشاء Embeddings بنجاح


In [75]:
def search_knowledge(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        knowledge_embeddings
    )[0]

    # اختيار أفضل صيغة لكل مشكلة، دون تكرار المشكلة نفسها
    best_by_problem = {}

    for idx, score in enumerate(scores):
        item_idx = query_to_item[idx]

        if (
            item_idx not in best_by_problem
            or score > best_by_problem[item_idx]["similarity"]
        ):
            best_by_problem[item_idx] = {
                "problem": knowledge_base[item_idx]["problem"],
                "solution": knowledge_base[item_idx]["solution"],
                "similarity": float(score)
            }

    results = sorted(
        best_by_problem.values(),
        key=lambda x: x["similarity"],
        reverse=True
    )[:top_k]

    for result in results:
        result["similarity"] = round(result["similarity"], 3)

    return results


results = search_knowledge(
    "رمز التحقق ما يوصلني على جوالي وما أقدر أدخل"
)

for i, result in enumerate(results, start=1):
    print(f"\nالنتيجة {i}")
    print("المشكلة:", result["problem"])
    print("درجة التشابه:", result["similarity"])
    print("الحل:", result["solution"])


النتيجة 1
المشكلة: رمز المصادقة متعددة العوامل MFA لا يصل إلى الهاتف
درجة التشابه: 0.315
الحل: تحقق من اتصال الهاتف بالشبكة ومن إعدادات تطبيق المصادقة. لا تشارك رمز التحقق مع أي شخص. إذا استمرت المشكلة، تواصل مع الدعم الفني.

النتيجة 2
المشكلة: المستخدم لا يستطيع تسجيل الدخول بسبب نسيان كلمة المرور
درجة التشابه: 0.289
الحل: استخدم خيار إعادة تعيين كلمة المرور من البوابة الرسمية. إذا استمرت المشكلة، تواصل مع فريق الدعم للتحقق من الحساب.


In [76]:
knowledge_base = [
    {
        "problem": "مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول",
        "queries": [
            "المستخدم لا يستطيع تسجيل الدخول بسبب نسيان كلمة المرور",
            "نسيت الباسورد ومش عارف أدخل",
            "مش قادر أدخل على حسابي",
            "كلمة المرور لا تعمل"
        ],
        "solution": "استخدم خيار إعادة تعيين كلمة المرور من البوابة الرسمية. إذا استمرت المشكلة، تواصل مع فريق الدعم للتحقق من الحساب."
    },
    {
        "problem": "رمز MFA أو رمز التحقق لا يصل إلى الهاتف",
        "queries": [
            "رمز المصادقة متعددة العوامل MFA لا يصل إلى الهاتف",
            "رمز التحقق ما يوصلني على جوالي",
            "كود الـ MFA مش بيوصل على الموبايل",
            "لا أستطيع تسجيل الدخول لأن رمز التحقق لا يصل"
        ],
        "solution": "تحقق من اتصال الهاتف بالشبكة ومن إعدادات تطبيق المصادقة. لا تشارك رمز التحقق مع أي شخص. إذا استمرت المشكلة، تواصل مع الدعم الفني."
    },
    {
        "problem": "مشكلة الاتصال بشبكة Wi-Fi",
        "queries": [
            "الاتصال بشبكة Wi-Fi الخاصة بالشركة لا يعمل",
            "الواي فاي مش شغال",
            "مش قادر أتصل بشبكة الشركة",
            "الإنترنت لا يعمل على اللابتوب"
        ],
        "solution": "تحقق من تفعيل Wi-Fi واختيار الشبكة الصحيحة. جرّب إعادة الاتصال، وإذا استمرت المشكلة زوّد الدعم باسم الشبكة ورسالة الخطأ."
    },
    {
        "problem": "الطابعة لا تطبع",
        "queries": [
            "الطابعة لا تطبع أو تظهر رسالة خطأ",
            "البرنتر مش بيطبع",
            "الورق محشور في الطابعة",
            "الطابعة تظهر خطأ عند الطباعة"
        ],
        "solution": "تحقق من تشغيل الطابعة واتصالها، ومن وجود الورق وعدم انحشاره. إذا استمرت المشكلة، أرسل للدعم رسالة الخطأ واسم الطابعة."
    },
    {
        "problem": "مشكلة وصول رسائل البريد الإلكتروني",
        "queries": [
            "رسائل البريد الإلكتروني لا تصل إلى Outlook",
            "الإيميلات مش بتوصلني",
            "مش بيجيلي بريد على أوتلوك",
            "لا تصلني رسائل البريد الإلكتروني"
        ],
        "solution": "تحقق من اتصال الإنترنت وحالة صندوق البريد، ثم أعد تشغيل التطبيق. إذا استمرت المشكلة، تواصل مع الدعم مع ذكر وقت حدوثها."
    },
    {
        "problem": "الاشتباه برسالة بريد إلكتروني تصيدية",
        "queries": [
    "الاشتباه برسالة بريد إلكتروني تصيدية",
    "وصلني إيميل مشبوه",
    "وصلني إيميل غريب فيه رابط يطلب كلمة المرور",
    "وصلني بريد إلكتروني يحتوي على رابط يطلب بيانات الدخول",
    "رسالة تطلب مني إدخال كلمة المرور عبر رابط",
    "إيميل فيه رابط مشبوه يطلب معلومات حسابي",
    "هل الرسالة دي محاولة احتيال",
    "رسالة فيها رابط غريب وتطلب بياناتي"
],
        "solution": "لا تضغط على الروابط أو تفتح المرفقات، ولا تدخل بياناتك. أبلغ فريق الأمن عبر القناة الرسمية واتبع سياسة المؤسسة."
    }
]

# إعادة بناء متجهات البحث من الصيغ البديلة
knowledge_texts = [
    query
    for item in knowledge_base
    for query in item["queries"]
]

# ربط كل صيغة بالحل المقابل لها
query_to_item = [
    item_index
    for item_index, item in enumerate(knowledge_base)
    for _ in item["queries"]
]

knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    normalize_embeddings=True
)

print("عدد المشكلات:", len(knowledge_base))
print("عدد صيغ البحث:", len(knowledge_texts))
print("تم تحديث قاعدة المعرفة")

عدد المشكلات: 6
عدد صيغ البحث: 28
تم تحديث قاعدة المعرفة


In [77]:
test_queries = [
    "نسيت كلمة المرور وما أقدر أدخل حسابي",
    "الواي فاي في الشركة ما يشتغل",
    "الطابعة ما تطبع والورق عالق",
    "الإيميلات ما توصلني في أوتلوك",
    "وصلني إيميل غريب فيه رابط يطلب كلمة المرور"
]

for query in test_queries:
    results = search_knowledge(query, top_k=1)

    print("\nالعبارة:", query)
    print("المشكلة المطابقة:", results[0]["problem"])
    print("درجة التشابه:", results[0]["similarity"])


العبارة: نسيت كلمة المرور وما أقدر أدخل حسابي
المشكلة المطابقة: مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول
درجة التشابه: 0.852

العبارة: الواي فاي في الشركة ما يشتغل
المشكلة المطابقة: مشكلة الاتصال بشبكة Wi-Fi
درجة التشابه: 0.719

العبارة: الطابعة ما تطبع والورق عالق
المشكلة المطابقة: الطابعة لا تطبع
درجة التشابه: 0.926

العبارة: الإيميلات ما توصلني في أوتلوك
المشكلة المطابقة: الاشتباه برسالة بريد إلكتروني تصيدية
درجة التشابه: 0.743

العبارة: وصلني إيميل غريب فيه رابط يطلب كلمة المرور
المشكلة المطابقة: الاشتباه برسالة بريد إلكتروني تصيدية
درجة التشابه: 1.0


In [78]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(knowledge_texts)

print("تم تجهيز البحث النصي.")
print("حجم مصفوفة البحث:", tfidf_matrix.shape)

تم تجهيز البحث النصي.
حجم مصفوفة البحث: (28, 1806)


In [79]:
def hybrid_search_knowledge(query, top_k=3, alpha=0.5):
    # البحث الدلالي
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )
    semantic_scores = cosine_similarity(
        query_embedding,
        knowledge_embeddings
    )[0]

    # البحث النصي
    query_tfidf = tfidf_vectorizer.transform([query])
    lexical_scores = cosine_similarity(
        query_tfidf,
        tfidf_matrix
    )[0]

    # دمج الدرجات
    combined_scores = (
        alpha * semantic_scores +
        (1 - alpha) * lexical_scores
    )

    # الاحتفاظ بأفضل درجة لكل مشكلة
    best_by_problem = {}

    for idx, score in enumerate(combined_scores):
        item_idx = query_to_item[idx]

        if (
            item_idx not in best_by_problem
            or score > best_by_problem[item_idx]["similarity"]
        ):
            best_by_problem[item_idx] = {
                "problem": knowledge_base[item_idx]["problem"],
                "solution": knowledge_base[item_idx]["solution"],
                "similarity": float(score)
            }

    results = sorted(
        best_by_problem.values(),
        key=lambda x: x["similarity"],
        reverse=True
    )[:top_k]

    for result in results:
        result["similarity"] = round(result["similarity"], 3)

    return results

In [80]:
test_queries = [
    "الإيميلات ما توصلني في أوتلوك",
    "وصلني إيميل غريب فيه رابط يطلب كلمة المرور"
]

for query in test_queries:
    results = hybrid_search_knowledge(query, top_k=3)

    print("\nالعبارة:", query)

    for result in results:
        print(
            result["problem"],
            "| درجة التشابه:", result["similarity"]
        )


العبارة: الإيميلات ما توصلني في أوتلوك
مشكلة وصول رسائل البريد الإلكتروني | درجة التشابه: 0.577
الاشتباه برسالة بريد إلكتروني تصيدية | درجة التشابه: 0.496
رمز MFA أو رمز التحقق لا يصل إلى الهاتف | درجة التشابه: 0.349

العبارة: وصلني إيميل غريب فيه رابط يطلب كلمة المرور
الاشتباه برسالة بريد إلكتروني تصيدية | درجة التشابه: 1.0
مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول | درجة التشابه: 0.483
مشكلة وصول رسائل البريد الإلكتروني | درجة التشابه: 0.222


In [81]:
query = "وصلني إيميل غريب فيه رابط يطلب كلمة المرور"

query_tfidf = tfidf_vectorizer.transform([query])
lexical_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

best_by_problem = {}

for idx, score in enumerate(lexical_scores):
    item_idx = query_to_item[idx]

    if (
        item_idx not in best_by_problem
        or score > best_by_problem[item_idx]["score"]
    ):
        best_by_problem[item_idx] = {
            "problem": knowledge_base[item_idx]["problem"],
            "score": float(score)
        }

for result in sorted(
    best_by_problem.values(),
    key=lambda x: x["score"],
    reverse=True
):
    print(result["problem"], "| درجة التشابه:", round(result["score"], 3))

الاشتباه برسالة بريد إلكتروني تصيدية | درجة التشابه: 1.0
مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول | درجة التشابه: 0.266
مشكلة وصول رسائل البريد الإلكتروني | درجة التشابه: 0.139
رمز MFA أو رمز التحقق لا يصل إلى الهاتف | درجة التشابه: 0.074
الطابعة لا تطبع | درجة التشابه: 0.043
مشكلة الاتصال بشبكة Wi-Fi | درجة التشابه: 0.023


In [82]:
query = "جاني بريد يقول إن حسابي بيتوقف إذا ما سجلت دخولي من الرابط"

results = hybrid_search_knowledge(query, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\nالنتيجة {i}")
    print("المشكلة:", result["problem"])
    print("درجة التشابه:", result["similarity"])


النتيجة 1
المشكلة: الاشتباه برسالة بريد إلكتروني تصيدية
درجة التشابه: 0.457

النتيجة 2
المشكلة: مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول
درجة التشابه: 0.384

النتيجة 3
المشكلة: مشكلة وصول رسائل البريد الإلكتروني
درجة التشابه: 0.311


In [83]:
test_cases = [
    {
        "query": "ما أتذكر الباسورد وكل ما أحاول أدخل يرفض",
        "expected": "مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول"
    },
    {
        "query": "الشبكة اللاسلكية في المكتب تفصل باستمرار",
        "expected": "مشكلة الاتصال بشبكة Wi-Fi"
    },
    {
        "query": "جهاز الطباعة يسحب الورق لكنه ما يطبع",
        "expected": "الطابعة لا تطبع"
    },
    {
        "query": "صندوق الوارد عندي فاضي رغم أن زملائي أرسلوا لي رسائل",
        "expected": "مشكلة وصول رسائل البريد الإلكتروني"
    },
    {
        "query": "وصلني رابط يهدد بإغلاق حسابي ويطلب بياناتي",
        "expected": "الاشتباه برسالة بريد إلكتروني تصيدية"
    },
    {
        "query": "رمز الدخول الإضافي ما وصلني على الجوال",
        "expected": "رمز MFA أو رمز التحقق لا يصل إلى الهاتف"
    }
]

print("عدد عبارات الاختبار:", len(test_cases))

عدد عبارات الاختبار: 6


In [84]:
correct = 0
evaluation_results = []

for case in test_cases:
    results = hybrid_search_knowledge(case["query"], top_k=1)
    predicted = results[0]["problem"]
    expected = case["expected"]
    is_correct = predicted == expected

    if is_correct:
        correct += 1

    evaluation_results.append({
        "العبارة": case["query"],
        "المشكلة المتوقعة": expected,
        "المشكلة المتوقعة من البحث": predicted,
        "صحيحة؟": "نعم" if is_correct else "لا",
        "درجة التشابه": results[0]["similarity"]
    })

import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df)

accuracy = correct / len(test_cases) * 100
print(f"\nالنتائج الصحيحة: {correct} من {len(test_cases)}")
print(f"دقة البحث على مجموعة الاختبار: {accuracy:.1f}%")

,العبارة,المشكلة المتوقعة,المشكلة المتوقعة من البحث,صحيحة؟,درجة التشابه
0,ما أتذكر الباسورد وكل ما أحاول أدخل يرفض,مشكلة نسيان كلمة المرور أو عدم القدرة على تسجي...,مشكلة نسيان كلمة المرور أو عدم القدرة على تسجي...,نعم,0.396
1,الشبكة اللاسلكية في المكتب تفصل باستمرار,مشكلة الاتصال بشبكة Wi-Fi,مشكلة الاتصال بشبكة Wi-Fi,نعم,0.306
2,جهاز الطباعة يسحب الورق لكنه ما يطبع,الطابعة لا تطبع,الطابعة لا تطبع,نعم,0.603
3,صندوق الوارد عندي فاضي رغم أن زملائي أرسلوا لي...,مشكلة وصول رسائل البريد الإلكتروني,مشكلة وصول رسائل البريد الإلكتروني,نعم,0.328
4,وصلني رابط يهدد بإغلاق حسابي ويطلب بياناتي,الاشتباه برسالة بريد إلكتروني تصيدية,الاشتباه برسالة بريد إلكتروني تصيدية,نعم,0.575
5,رمز الدخول الإضافي ما وصلني على الجوال,رمز MFA أو رمز التحقق لا يصل إلى الهاتف,رمز MFA أو رمز التحقق لا يصل إلى الهاتف,نعم,0.634



النتائج الصحيحة: 6 من 6
دقة البحث على مجموعة الاختبار: 100.0%


In [85]:
def get_ticket_assistance(query, threshold=0.35):
    results = hybrid_search_knowledge(query, top_k=1)
    best_match = results[0]

    if best_match["similarity"] < threshold:
        return {
            "وصف المشكلة": query,
            "المشكلة الأقرب": "لم يتم العثور على مشكلة مطابقة بدرجة كافية",
            "الحل المقترح": (
                "ننصح بالتواصل مع فريق الدعم الفني "
                "لمراجعة المشكلة وتحديد الحل المناسب."
            ),
            "درجة التشابه": best_match["similarity"],
            "حالة المطابقة": "تحتاج إلى مراجعة"
        }

    return {
        "وصف المشكلة": query,
        "المشكلة الأقرب": best_match["problem"],
        "الحل المقترح": best_match["solution"],
        "درجة التشابه": best_match["similarity"],
        "حالة المطابقة": "مطابقة مبدئية"
    }

In [86]:
test_query = "جهازي يصدر صوتًا غريبًا عند التشغيل"

result = get_ticket_assistance(test_query)

for key, value in result.items():
    print(f"{key}: {value}")

وصف المشكلة: جهازي يصدر صوتًا غريبًا عند التشغيل
المشكلة الأقرب: لم يتم العثور على مشكلة مطابقة بدرجة كافية
الحل المقترح: ننصح بالتواصل مع فريق الدعم الفني لمراجعة المشكلة وتحديد الحل المناسب.
درجة التشابه: 0.292
حالة المطابقة: تحتاج إلى مراجعة


In [87]:
%pip install -q gradio

In [88]:
import gradio as gr

def respond_to_ticket(query):
    if not query or not query.strip():
        return "", "", "", "", ""

    query = query.strip()

    predicted_category = model.predict([query])[0]
    assistance = get_ticket_assistance(query)

    return (
        predicted_category,
        assistance["المشكلة الأقرب"],
        assistance["الحل المقترح"],
        str(assistance["درجة التشابه"]),
        assistance["حالة المطابقة"]
    )


with gr.Blocks(title="Smart Ticket Assistant") as demo:
    gr.Markdown("# Smart Ticket Assistant")
    gr.Markdown(
        "مساعد لتصنيف مشكلات الدعم الفني "
        "واسترجاع الحلول من قاعدة المعرفة."
    )

    ticket_input = gr.Textbox(
        label="وصف المشكلة",
        placeholder="مثال: رمز التحقق ما يوصلني على الجوال",
        lines=3
    )

    submit_btn = gr.Button("تحليل المشكلة", variant="primary")

    category_output = gr.Textbox(label="فئة التذكرة المتوقعة")
    matched_problem = gr.Textbox(label="المشكلة الأقرب")
    suggested_solution = gr.Textbox(label="الحل المقترح", lines=3)
    similarity_score = gr.Textbox(label="درجة التشابه")
    match_status = gr.Textbox(label="حالة المطابقة")

    submit_btn.click(
        fn=respond_to_ticket,
        inputs=ticket_input,
        outputs=[
            category_output,
            matched_problem,
            suggested_solution,
            similarity_score,
            match_status
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ae5ce1dcdc5c2b49b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [89]:
for case in test_cases:
    results = hybrid_search_knowledge(case["query"], top_k=1)
    best = results[0]

    print("الاستفسار:", case["query"])
    print("المشكلة المتوقعة:", case["expected"])
    print("المشكلة المسترجعة:", best["problem"])
    print("درجة التشابه:", best["similarity"])
    print("-" * 50)

الاستفسار: ما أتذكر الباسورد وكل ما أحاول أدخل يرفض
المشكلة المتوقعة: مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول
المشكلة المسترجعة: مشكلة نسيان كلمة المرور أو عدم القدرة على تسجيل الدخول
درجة التشابه: 0.396
--------------------------------------------------
الاستفسار: الشبكة اللاسلكية في المكتب تفصل باستمرار
المشكلة المتوقعة: مشكلة الاتصال بشبكة Wi-Fi
المشكلة المسترجعة: مشكلة الاتصال بشبكة Wi-Fi
درجة التشابه: 0.306
--------------------------------------------------
الاستفسار: جهاز الطباعة يسحب الورق لكنه ما يطبع
المشكلة المتوقعة: الطابعة لا تطبع
المشكلة المسترجعة: الطابعة لا تطبع
درجة التشابه: 0.603
--------------------------------------------------
الاستفسار: صندوق الوارد عندي فاضي رغم أن زملائي أرسلوا لي رسائل
المشكلة المتوقعة: مشكلة وصول رسائل البريد الإلكتروني
المشكلة المسترجعة: مشكلة وصول رسائل البريد الإلكتروني
درجة التشابه: 0.328
--------------------------------------------------
الاستفسار: وصلني رابط يهدد بإغلاق حسابي ويطلب بياناتي
المشكلة المتوقعة: الاشتباه برسالة 